# Chapter 38 — Capstone extensions: data-quality workbench and student-performance analytics

**Book role:** Capstone extension  
**Suggested book space:** about 10 pages after adding diagrams, annotated screenshots, worked examples, and review exercises.  
**How to study:** read the explanations, run one code cell at a time, change the inputs, then complete the lab without copying the solution first.

> **Execution note:** Tkinter opens native desktop windows. Run GUI cells in a local Python/Jupyter session with a working display, not a headless notebook server. Close each window before moving to the next GUI example. Do not use **Run All** on the complete book: some cells open windows, read or write files, or demonstrate database operations.

## Learning outcomes

- Extend the architecture to a data-quality monitoring workflow.
- Apply the same import/profile/report pattern to student-performance data.
- Choose charts and metrics that respect the data grain and privacy constraints.
- Design reusable components rather than duplicating the sales app.

## 1. Quality workbench as a reusable product pattern

A data-quality workbench uses the same high-level pipeline as the sales explorer: load a source, infer/validate schema, calculate a profile, inspect issues, apply controlled cleaning rules, and export a report. The domain logic changes, but file dialogs, status messages, table view, error reporting, and export boundaries can be reusable components. Make the rule engine independent of widgets and attach rule IDs/severity to each result.

For a source with many columns, a quality profile can show missingness, uniqueness, dtype, approximate range, and category cardinality. The UI should support selecting a rule and viewing affected rows without loading every row into a widget. For very large datasets, profile summaries and drill-down queries may run at the source rather than in one huge in-memory DataFrame.

### Worked example 1: Create a generic data-quality profile

A generic profile reports structural characteristics but cannot infer business meaning alone. Domain rules must specify plausible score ranges, assessment maximums, weighting, and allowed categories.

In [ ]:
import pandas as pd

def profile_dataframe(frame: pd.DataFrame) -> pd.DataFrame:
    total = max(len(frame), 1)
    return pd.DataFrame({
        "column": frame.columns,
        "dtype": [str(frame[name].dtype) for name in frame.columns],
        "missing_count": [int(frame[name].isna().sum()) for name in frame.columns],
        "missing_pct": [round(float(frame[name].isna().mean() * 100), 2) for name in frame.columns],
        "distinct_count": [int(frame[name].nunique(dropna=True)) for name in frame.columns],
    })

example = pd.DataFrame({"student_id": ["S1", "S2", "S3"], "score": [85, None, 91], "subject": ["Math", "Math", "Science"]})
print(profile_dataframe(example))

## 2. Student analytics requires careful metric interpretation

A student-performance dataset may include student ID, class/section, assessment, subject, score, maximum score, attendance, and term. Derived measures such as percentage score depend on assessment maximums and weighting. Averages across subjects are only meaningful if the scoring scales or weighting rules are defined. Do not compare raw marks from assessments with different maximum scores as if they were equivalent.

Student data is sensitive. Use synthetic or anonymized examples, minimize identifying fields, and avoid displaying named individual rankings by default. Aggregate summaries can reveal useful patterns, but small-group breakdowns can expose individuals. Access, export, retention, and screenshot policies must reflect institutional rules and applicable law. A learning notebook should not embed real student records or credentials.

### Worked example 2: Define a normalized assessment score

This example normalizes scores but flags values outside the maximum or with invalid denominators. Weighting assessment percentages into a final grade is a separate policy decision that should be written down.

In [ ]:
import pandas as pd

def add_score_percentage(frame: pd.DataFrame) -> pd.DataFrame:
    required = {"score", "max_score"}
    if not required.issubset(frame.columns):
        raise ValueError(f"Required columns: {sorted(required)}")
    result = frame.copy()
    result["score"] = pd.to_numeric(result["score"], errors="coerce")
    result["max_score"] = pd.to_numeric(result["max_score"], errors="coerce")
    invalid = result["max_score"].isna() | result["max_score"].le(0) | result["score"].lt(0) | result["score"].gt(result["max_score"])
    result["quality_issue"] = invalid
    result["score_pct"] = (result["score"] / result["max_score"] * 100).where(~invalid)
    return result

scores = pd.DataFrame({"score": [18, 40, 105, None], "max_score": [20, 50, 100, 100]})
print(add_score_percentage(scores))

## 3. Reusable application components

Reusable components might include `FileImportService`, `SchemaValidator`, `QualityProfileService`, `FilterPanel`, `DataTableView`, `KpiPanel`, `ChartPanel`, and `ExportService`. Reuse should occur where behavior is stable across domains; do not force every dataset into a `sales` schema. A generic DataFrame explorer can display fields and profiles, while domain-specific services define score weighting or profit calculations.

A clean extension demonstrates that the architecture was not overfit to the first sample. Add a new dataset adapter and service without altering unrelated UI internals. Keep domain labels in a data dictionary and let views receive configuration for titles, formats, and units. Test the quality and analytics rules with domain-specific fixtures.

## Guided lab: Lab: build one reusable extension

Choose either a generic Data Quality Workbench or a synthetic Student Performance Analyzer. Reuse the same file-import, table, quality, and export components but implement domain-specific rules in separate services. Include a data dictionary, at least ten tests, privacy-conscious displays/exports, and a clear explanation of which metrics are raw, normalized, or weighted.

## Review questions and practice

1. Why is a generic missingness profile insufficient as a complete quality assessment?
2. Why should marks with different maximums be normalized before some comparisons?
3. Name three reusable UI/service components across sales and quality workflows.
4. What privacy concerns apply to student-level data?

## Further reading

- pandas profiling primitives: https://pandas.pydata.org/docs/user_guide/
- NIST privacy framework: https://www.nist.gov/privacy-framework

## Notebook checkpoint

Before moving on, write a short note in your own words explaining one design decision from this chapter, one failure mode to test, and one way the idea will be used in a data-analysis application.